# 21 — Exploratory Data Analysis for Classification (Objective 2)

**Objective 2:** *"To build a model that estimates whether a warehouse will report a breakdown"* —
*"estimating breakdown risk so that preventive action can be planned."* Classification algorithms from logistic
regression to CatBoost are to be evaluated with accuracy, precision, recall, F1, ROC-AUC and PR-AUC.

Before any model is built, this notebook examines the data from the point of view of that task. The target is a
**binary label**: **Not High Risk** (0–3 breakdowns) vs **High Risk** (4+ breakdowns), with the threshold chosen
on the basis of §3's evidence.

| § | Question | Analysis |
|---|---|---|
| 1 | What does the binary target look like, and is the 4+ threshold justified? | univariate — target |
| 2 | Which columns may be used to predict it, and which may not? | roles |
| 3 | Do the features change across breakdown counts in a way the binary threshold respects? | target vs features |
| 4 | What must scale-sensitive models and a stratified split cope with? | univariate — features |
| 5 | Which numeric features differ between risk classes? | bivariate — ANOVA, Kruskal–Wallis |
| 6 | Which categorical and 0/1 features differ between risk classes? | bivariate — chi-square |
| 7 | Do features repeat each other, and do the classes occupy different regions of the data? | multivariate |
| 8 | What does this mean for the classification design? | open items |

**Input:** `data/preprocessed/warehouse_preprocessed.csv`
**Output:** none. This notebook reports only; it writes nothing.

The binary threshold is set at 4+ breakdowns on the basis of §3's evidence; every section from §3.3 onward uses this
target. Further open items for model-building are set out in §8, to be resolved in the transformation and feature
engineering steps.

**A method note.** This analysis uses all 25,000 warehouses; the test set is separated afterwards, in the data
transformation step. No model is fitted and nothing is learned from the data here. Decisions in later notebooks that
*select* features on this evidence are made or checked again on the training split before they are applied, so the
held-back test set does not quietly shape the model.

## 0. Setup

The two binary risk classes are **Not High Risk** (0–3 breakdowns) and **High Risk** (4–6 breakdowns). Charts use
a light-to-dark hue pair and always label both classes by name.

In [ ]:
import sys, pathlib

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *

set_style()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

CLASSES = ["Not High Risk", "High Risk"]
CLASS_COLOURS = {"Not High Risk": "#86b6ef", "High Risk": "#104281"}
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]
DIVERGING = LinearSegmentedColormap.from_list("below_same_above", ["#2a78d6", "#f0efec", "#e34948"])

df = load_preprocessed()
print(f"loaded {df.shape[0]:,} rows x {df.shape[1]} columns from {PREPROCESSED_FILE.name}")

---
## 1. The target — breakdown risk

`wh_breakdown_l3m` counts the breakdowns each warehouse faced in the last three months. The binary risk label
is defined at a threshold of 4+ breakdowns:

| Class | Breakdowns in 3 months | Warehouses |
|---|---|---|
| Not High Risk | 0–3 | 13,026 (52.1%) |
| High Risk | 4–6 | 11,974 (47.9%) |

The threshold is chosen on the basis of §3.2's evidence: the 3|4 step is the largest single η² step across all three
features that relate to breakdowns. This section creates the label, confirms every count lands in exactly one class,
and reports class balance.

**Why balance matters for a classifier.** A model trained on very unequal classes can score high accuracy by favouring the
largest class, while missing the small one — which, for risk, is usually the class that matters.

In [ ]:
CLASSES = ["Not High Risk", "High Risk"]
TARGET_BINS = [-0.5, 3.5, 6.5]

df["breakdown_risk"] = pd.cut(df["wh_breakdown_l3m"], bins=TARGET_BINS,
                              labels=CLASSES, ordered=True)
assert df["breakdown_risk"].notna().all()

print("breakdown count against assigned class — each count falls in exactly one class:")
print(pd.crosstab(df["wh_breakdown_l3m"], df["breakdown_risk"]).to_string())

counts = df["breakdown_risk"].value_counts().sort_index()
pct = (100 * counts / len(df)).round(2)
print("\nclass sizes:")
print(pd.DataFrame({"warehouses": counts, "pct": pct}).to_string())

In [ ]:
raw = df["wh_breakdown_l3m"].value_counts().sort_index()
raw_class = pd.cut(raw.index, bins=TARGET_BINS, labels=CLASSES)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [7, 3]})
axes[0].bar(raw.index, raw.to_numpy(), color=[CLASS_COLOURS[c] for c in raw_class], width=0.8)
axes[0].set_xticks(raw.index)
axes[0].set_xlabel("breakdowns in the last 3 months"); axes[0].set_ylabel("warehouses")
axes[0].set_title("Breakdown count, coloured by the class it falls in", fontweight="bold")
axes[0].legend(handles=[Patch(color=CLASS_COLOURS[c], label=c) for c in CLASSES], frameon=False)

axes[1].bar(CLASSES, counts.to_numpy(), color=[CLASS_COLOURS[c] for c in CLASSES], width=0.6)
for i, v in enumerate(counts.to_numpy()):
    axes[1].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9, color="#0b0b0b")
axes[1].set_ylabel("warehouses")
axes[1].set_title("Binary risk classes", fontweight="bold")
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> - The crosstab confirms the banding is clean: counts 0–3 fall only in Not High Risk, counts 4–6 only in High Risk.
>
> - **The two classes are nearly balanced.** Not High Risk holds 13,026 (52.1%) warehouses and High Risk holds
>   11,974 (47.9%) — a largest-to-smallest ratio of **1.09 : 1**. The two classes are nearly balanced, so accuracy
>   is a safe primary metric and no resampling is expected to be necessary.
>
> - For comparison, a binary *any breakdown* split puts 96.37% in one class (26.53 : 1) — almost entirely the
>   908 warehouses with zero breakdowns — making it analytically useless.
>
> - Balance, however, is only half of what a target needs. The classes must also *differ* in ways a model can detect —
>   §3 checks that.

---
## 2. Which columns may predict breakdown risk?

Every column is given one role before any relationship with the target is measured, so that no column is chosen *because*
it predicts well. The roles follow from each column's definition in the data dictionary:

| Role | Columns | Why |
|---|---|---|
| **Row key** | `Ware_house_ID` | identifies a warehouse; never a feature |
| **Target source** | `wh_breakdown_l3m` | the target is built from it. Leaving it among the features would hand the model the answer, so it is removed once `breakdown_risk` exists |
| **Operational measures recorded over a period** | `product_wg_ton`, `storage_issue_reported_l3m`, `num_refill_req_l3m`, `govt_check_l3m` (last 3 months); `transport_issue_l1y` (last year) | counted over the same window as the breakdowns, or one that contains it |
| **Warehouse characteristics** | numeric: `wh_est_year`, `workers_num`, `dist_from_hub`, `Competitor_in_mkt`, `retail_shop_num`, `distributor_num`; 0/1: `electric_supply`, `temp_reg_mach`, `flood_proof`, `flood_impacted`, `is_unrated_warehouse`; categorical: `approved_wh_govt_certificate`, `Location_type`, `WH_capacity_size`, `WH_regional_zone`, `zone`, `wh_owner_type` | describe what the warehouse *is* — site, size, infrastructure, age, market, certification |
| **Recording flag** | `wh_est_year_missing` | says whether the year was recorded |

**Why the period measures are listed separately.** This section applies a model that estimates risk *so that preventive
action can be planned*. Characteristics are known before a quarter begins. The period measures are counted during the very
three months whose breakdowns are being predicted, so they may partly describe the same disruption rather than warn of it.
Whether they are used is **not decided here** — §5 and §8 measure how much of the signal depends on them.


In [ ]:
key = "Ware_house_ID"
target_source = "wh_breakdown_l3m"
period_measures = ["product_wg_ton", "storage_issue_reported_l3m", "num_refill_req_l3m",
                   "govt_check_l3m", "transport_issue_l1y"]
characteristics_numeric = ["wh_est_year", "workers_num", "dist_from_hub", "Competitor_in_mkt",
                           "retail_shop_num", "distributor_num"]
characteristics_binary = ["electric_supply", "temp_reg_mach", "flood_proof", "flood_impacted", "is_unrated_warehouse"]
characteristics_categorical = ["approved_wh_govt_certificate", "Location_type", "WH_capacity_size",
                               "WH_regional_zone", "zone", "wh_owner_type"]
recording_flags = ["wh_est_year_missing"]

assigned = ([key, target_source] + period_measures + characteristics_numeric + characteristics_binary
            + characteristics_categorical + recording_flags)
original_columns = [c for c in df.columns if c != "breakdown_risk"]
assert sorted(assigned) == sorted(original_columns), set(assigned) ^ set(original_columns)
assert len(assigned) == len(set(assigned)), "a column was assigned twice"

numeric_features = period_measures + characteristics_numeric
binary_features = characteristics_binary + recording_flags
categorical_features = characteristics_categorical
candidates = numeric_features + binary_features + categorical_features

for name, cols in [("row key", [key]), ("target source", [target_source]),
                   ("period measures", period_measures), ("characteristics - numeric", characteristics_numeric),
                   ("characteristics - 0/1", characteristics_binary),
                   ("characteristics - categorical", characteristics_categorical), ("recording flag", recording_flags)]:
    print(f"{name:<32} {len(cols):>2}")
print(f"{'total':<32} {len(assigned):>2}  of {len(original_columns)} columns")
print(f"\ncandidate features: {len(candidates)}")

> **Interpretation.**
>
> - All 25 columns carry exactly one role, and the assertions guarantee none is left out or counted twice.
>   Removing the row key and the target source leaves **23 candidate features**: 5 period measures, 6 numeric
>   characteristics, 5 on/off characteristics, 6 categorical characteristics and the recording flag.
>
> - Whether the period measures may be used is an open question. It is closed on review of the evidence in §5 and recorded
>   in §8.

---
## 3. Do the features change across breakdown counts — and where?

Section 1 checked the banding on balance alone. A banding is also only useful if warehouses in different classes actually
*differ*. If some feature rose steadily from 0 to 6 breakdowns, the class boundaries would cut a gradient that a model can
follow. If it rose to 4 and then stayed flat, a model would struggle to tell Medium from High on that feature.

For every numeric candidate: its Spearman correlation with the raw breakdown count — across all warehouses, and across
rated warehouses only, because every one of the 908 unrated warehouses sits at zero breakdowns (NB 00 §10) — and its
median at each count from 0 to 6. `wh_est_year` is measured on **recorded years only** (NB 01 §4 filled missing years at
2009, which would pull every count toward the same value).

The chart then follows the features whose correlation with the count reaches **0.10 in absolute value**, a threshold fixed
before looking. It shows each feature's mean at every count in standard deviations from its overall mean, so that tons and
counts share one axis. The vertical lines mark the class boundaries.

In [ ]:
rated = df["is_unrated_warehouse"] == 0
recorded_year = df["wh_est_year_missing"] == 0

rows = []
for c in numeric_features:
    use = recorded_year if c == "wh_est_year" else pd.Series(True, index=df.index)
    d = df[use]
    r = d["is_unrated_warehouse"] == 0
    row = {"feature": c + (" (recorded years)" if c == "wh_est_year" else ""),
           "spearman_all": d[c].corr(d[target_source], method="spearman"),
           "spearman_rated_only": d.loc[r, c].corr(d.loc[r, target_source], method="spearman")}
    for k in range(7):
        row[f"median at {k}"] = d.loc[d[target_source] == k, c].median()
    rows.append(row)
by_count = pd.DataFrame(rows).set_index("feature")
by_count.round(3)

In [ ]:
followed = [c for c in numeric_features
            if abs(by_count.loc[c + (" (recorded years)" if c == "wh_est_year" else ""), "spearman_all"]) >= 0.10]
assert len(followed) <= len(SERIES), "more features than the fixed colour order allows — facet instead"
print(f"features reaching |Spearman| >= 0.10 with the breakdown count: {followed}\n")

fig, ax = plt.subplots(figsize=(11, 5.5))
for colour, c in zip(SERIES, followed):
    d = df[recorded_year] if c == "wh_est_year" else df
    means = d.groupby(target_source)[c].mean()
    z = (means - d[c].mean()) / d[c].std()
    ax.plot(z.index, z.to_numpy(), marker="o", markersize=7, linewidth=2, color=colour,
            label=c + (" (recorded years)" if c == "wh_est_year" else ""))
ax.axhline(0, color="#c3c2b7", linewidth=1)
ax.axvline(3.5, color="#898781", linewidth=1)   # binary threshold: 3|4
lo, hi = ax.get_ylim()
ax.set_ylim(lo, hi + 0.15 * (hi - lo))
for label, x in zip(CLASSES, (1.5, 5.0)):
    ax.text(x, hi + 0.06 * (hi - lo), label, ha="center", fontsize=10, fontweight="bold", color="#52514e")
ax.set_xticks(range(7))
ax.set_xlabel("breakdowns in the last 3 months")
ax.set_ylabel("mean, in standard deviations\nfrom the feature's overall mean")
ax.set_title("How the features that move with breakdowns change across the count", fontweight="bold")
ax.legend(frameon=False, loc="lower right")
plt.tight_layout(); plt.show()

print("mean of each followed feature at each breakdown count (recorded units):")
print(pd.DataFrame({c: (df[recorded_year] if c == "wh_est_year" else df).groupby(target_source)[c].mean()
                    for c in followed}).round(2).T.to_string())

> **Interpretation.**
>
> - **Only three of the eleven numeric candidates move with breakdowns.** Shipment weight (Spearman 0.339), storage issues
>   (0.350) and establishment year on recorded years (−0.380) all relate to the count, and all three keep most of that
>   relationship among rated warehouses alone (0.268, 0.273, −0.308). Every other numeric feature has a correlation of 0.017
>   or less, and its median barely moves across all seven counts: 28 workers, 4 refill requests, 3 competitors and 0
>   transport issues at every count.
>
> - **How the three change — read from the chart, confirmed by the table of means.** All three change sharply from 0 to 1
>   to 2 breakdowns. Mean shipment rises from 5,430 t to 11,627 t to 21,547 t, and mean establishment year falls from
>   2021.94 to 2017.30 to 2009.77. **From 2 to 3 they barely move** (21,547 t against 22,118 t; storage issues 16.86
>   against 17.29). **From 3 to 4 there is a meaningful step** — the largest single step among counts 2 and above
>   (22,118 t to 25,591 t). **From 4 to 6 they are flat**: 25,591, 25,229 and 25,307 t; storage issues 20.02, 19.75 and
>   19.82; year 2006.76, 2007.02 and 2006.95. The binary threshold at the 3|4 boundary is motivated by this pattern.
>
> - The point at 0 breakdowns is the 908 unrated warehouses — NB 00 §10 found that zero breakdowns and unrated status
>   describe exactly the same warehouses.

### 3.2 Where do adjacent counts actually differ — and where is the best binary threshold?

Two measurements, both using **η²** — the share of a feature's variance explained by a grouping (0 = the groups do not
differ at all).

**Step by step.** For each pair of neighbouring counts (0 and 1, 1 and 2, …), η² of each followed feature between *just
those two counts*. A value near zero means warehouses with k and k + 1 breakdowns are indistinguishable on that feature.
The step with the highest η² across all three features among counts 2 and above identifies the natural binary cutpoint.

**Signal kept.** η² of each feature under a banding can never exceed η² under the full seven-level count, because
merging counts can only hide differences, never create them. *Signal kept* = η² under the banding ÷ η² under the raw
count, averaged over the followed features, measures how much of the features' relationship with breakdowns a banding
preserves. It is reported for all warehouses and for rated warehouses only.

In [ ]:
rows = []
for k in range(6):
    row = {"step": f"{k} vs {k + 1}", "binary boundary": "yes" if k == 3 else ""}
    for c in followed:
        use = recorded_year if c == "wh_est_year" else pd.Series(True, index=df.index)
        pair = df[use & df[target_source].isin([k, k + 1])]
        _vals = pair[c]; _groups = pair[target_source]
        _gm = _vals.mean()
        _between = sum(len(v) * (v.mean() - _gm) ** 2 for _, v in _vals.groupby(_groups, observed=True))
        row[f"eta_sq: {c}"] = round(_between / ((_vals - _gm) ** 2).sum(), 4)
    rows.append(row)
print("how different are warehouses with k and k + 1 breakdowns? (eta squared between the two counts)")
print(pd.DataFrame(rows).set_index("step").to_string())

bandings = {
    "binary: 0-3 / 4-6": [-0.5, 3.5, 6.5],
    "binary: 0 / 1-6 (any breakdown)": [-0.5, 0.5, 6.5],
    "raw count, seven levels": [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5, 5.5, 6.5],
}

raw_eta = {}
for c in followed:
    use = recorded_year if c == "wh_est_year" else pd.Series(True, index=df.index)
    _vals = df.loc[use, c]; _groups = df.loc[use, target_source]
    _gm = _vals.mean()
    _between = sum(len(v) * (v.mean() - _gm) ** 2 for _, v in _vals.groupby(_groups, observed=True))
    raw_eta[c] = _between / ((_vals - _gm) ** 2).sum()

raw_eta_rated = {}
for c in followed:
    use = recorded_year if c == "wh_est_year" else pd.Series(True, index=df.index)
    _vals = df.loc[use & rated, c]; _groups = df.loc[use & rated, target_source]
    _gm = _vals.mean()
    _between = sum(len(v) * (v.mean() - _gm) ** 2 for _, v in _vals.groupby(_groups, observed=True))
    raw_eta_rated[c] = _between / ((_vals - _gm) ** 2).sum()

rows = []
for name, bins in bandings.items():
    y = pd.cut(df[target_source], bins=bins, labels=False)
    share = 100 * y.value_counts(normalize=True)
    lowest = y == 0
    _kept_all = []
    for c in followed:
        use = recorded_year if c == "wh_est_year" else pd.Series(True, index=df.index)
        _vals = df.loc[use, c]; _groups = y[use]
        _gm = _vals.mean()
        _between = sum(len(v) * (v.mean() - _gm) ** 2 for _, v in _vals.groupby(_groups, observed=True))
        _kept_all.append((_between / ((_vals - _gm) ** 2).sum()) / raw_eta[c])
    _kept_rated = []
    for c in followed:
        use = recorded_year if c == "wh_est_year" else pd.Series(True, index=df.index)
        _vals = df.loc[use & rated, c]; _groups = y[use & rated]
        _gm = _vals.mean()
        _between = sum(len(v) * (v.mean() - _gm) ** 2 for _, v in _vals.groupby(_groups, observed=True))
        _kept_rated.append((_between / ((_vals - _gm) ** 2).sum()) / raw_eta_rated[c])
    rows.append({
        "banding": name, "classes": y.nunique(),
        "smallest_class_pct": round(share.min(), 2),
        "largest_to_smallest": round(share.max() / share.min(), 2),
        "unrated_pct_of_lowest_class": round(100 * df.loc[lowest, "is_unrated_warehouse"].mean(), 1),
        "signal_kept_all_pct": round(100 * np.mean(_kept_all), 1),
        "signal_kept_rated_pct": round(100 * np.mean(_kept_rated), 1),
    })
print("\nbandings compared:")
pd.DataFrame(rows).set_index("banding")

> **Interpretation.**
>
> - **Where neighbouring counts differ.**
>
> | Step | η² — shipment / storage issues / year | Binary boundary |
> |---|---|---|
> | 0 vs 1 | 0.2095 / 0.5117 / 0.4031 | |
> | 1 vs 2 | 0.1546 / 0.1644 / 0.2112 | |
> | 2 vs 3 | 0.0006 / 0.0006 / 0.0004 | |
> | **3 vs 4** | **0.0235 / 0.0249 / 0.0351** | **yes** |
> | 4 vs 5 | 0.0003 / 0.0003 / 0.0004 | |
> | 5 vs 6 | 0.0000 / 0.0000 / 0.0000 | |
>
> - **The 3|4 step is the largest single η² step on all three features among the counts from 2 onward.**
>   Using it as the binary cutpoint preserves the most signal between classes. The steps at 0|1 and 1|2
>   are larger, but these involve the 908 unrated warehouses that sit at exactly zero breakdowns — a rule
>   already in the data rather than a meaningful separator for the operating network.
>
> - **The naive alternative — any breakdown vs none — is useless.** Binary *any breakdown* captures signal
>   only from the unrated group; among rated warehouses it keeps 0.0% of the signal. The 3|4 binary split
>   keeps 48.5% of the rated signal with a near-balanced 1.09 : 1 split.

### 3.3 Decision — the binary target

> **Decision — binary target.**
>
> The breakdown count is split into two classes at 4+ breakdowns:
> - **Not High Risk**: 0–3 breakdowns (13,026 warehouses, 52.1%)
> - **High Risk**: 4–6 breakdowns (11,974 warehouses, 47.9%)
>
> The 3|4 step carries the most signal across all three key features. The naive alternative — any breakdown vs
> none — gives a 96%/4% split that is entirely explained by the storage_issue == 0 rule, making it analytically
> useless. The 3|4 threshold avoids this while keeping the split near-balanced.

---
## 4. The features from a classifier's point of view

NB 00 §6–§8 described every column's distribution. Two questions are specific to this objective:

- **Scale.** Logistic regression, support vector machines and distance- or variance-based methods are sensitive to the
  units a feature is recorded in; tree-based models are not. How different are the scales?
- **Thin categories under a stratified split.** NB 22 holds back 20% of warehouses as a test set, stratified by class. A
  category level with few warehouses in one class leaves very few in that class's test portion, where performance
  cannot be judged. How thin do the rarest levels get?

`wh_est_year` appears here as NB 01 left it — filled — because that is the column a model receives.

In [ ]:
scale = pd.DataFrame({
    "min": df[numeric_features].min(),
    "max": df[numeric_features].max(),
    "std": df[numeric_features].std().round(3),
    "skew": df[numeric_features].skew().round(3),
}).sort_values("std", ascending=False)
print(f"largest standard deviation / smallest: {scale['std'].max() / scale['std'].min():,.0f} : 1\n")
print(scale.to_string())

binary_share = (100 * df[binary_features].mean()).round(2).rename("pct_equal_to_1")
print("\n0/1 features — share equal to 1:")
print(binary_share.to_string())

rows = []
for c in categorical_features:
    t = pd.crosstab(df[c], df["breakdown_risk"])
    rarest = df[c].value_counts().idxmin()
    row = {"feature": c, "levels": df[c].nunique(), "rarest_level": rarest, "rarest_n": int(t.loc[rarest].sum())}
    for k in CLASSES:
        row[f"rarest_in_{k}"] = int(t.loc[rarest, k])
    row["approx_in_smallest_test_share"] = round(0.2 * t.loc[rarest].min())
    rows.append(row)
thin = pd.DataFrame(rows).set_index("feature")
print(f"\none-hot encoding the {len(categorical_features)} categorical features would create "
      f"{thin['levels'].sum()} columns ({(thin['levels'] - 1).sum()} with one level of each dropped)\n")
thin

> **Interpretation.**
>
> - **Scale — scaling is required for some models, not others.** The largest standard deviation (shipment weight,
>   11,607.76) is **10,164 times** the smallest (competitors, 1.142). Unscaled, logistic regression's regularisation and
>   an SVM's distances would be dominated by shipment weight and retail shops purely because of their units. Tree-based
>   models split one feature at a time and are unaffected. Skew is mild everywhere, highest for transport issues (1.611)
>   and workers (1.079).
>
> - **On/off features.** Three are rare: flood-proofing (5.46%), the unrated flag (3.63%) and flood impact (9.82%). The
>   recording flag is the most even, at 47.52%.
>
> - **Thin categories under a stratified 20% test set.** With the binary target, most level–class combinations are
>   comfortably sized.
>
> - **`approved_wh_govt_certificate = Unrated` occurs only in Not High Risk** (all 908 unrated warehouses have
>   0 breakdowns). It marks those warehouses exactly and tells a model nothing about separating Not High Risk from
>   High Risk beyond that rule.
> - **`zone = East`** has 429 total warehouses; the split across the binary classes is well above the thin
>   threshold for a 20% test set.
> - The remaining rarest levels each have several hundred warehouses in every binary class.
>
> - **One-hot encoding** all six categorical features would add 23 columns (17 with one level of each dropped).

---
## 5. Numeric features against risk class

This section applies analysis of variance. Two tests per feature:

- **One-way ANOVA** — do the class means differ?
- **Kruskal–Wallis** — the same question without assuming normally distributed values, which matters for counts and for
  the flat distributions NB 00 §6 found.

With 25,000 warehouses almost any difference is significant, so the deciding figure is the effect size **η²**: the share of
a feature's variance explained by the class a warehouse is in. The rule of thumb reads 0.01 as small, 0.06 as medium and
0.14 as large.

Two refinements, because of what earlier notebooks found:

- **Rated warehouses only, alongside all warehouses.** The 908 unrated warehouses all fall in Not High Risk (§1) and sit
  at zero storage issues (NB 00 §10). NB 11 §5 showed that a block like this can create most of an association on its own.
- **`wh_est_year` twice** — on recorded years, which describe the real relationship, and filled, as a model receives it.
  The gap between the two rows shows what NB 01's filling costs this objective.

Spearman correlation with the binary class order (Not High Risk = 0, High Risk = 1) is added, because the classes are
ordered.

In [ ]:
class_order = df["breakdown_risk"].cat.codes

specs = [(c, c, pd.Series(True, index=df.index)) for c in numeric_features if c != "wh_est_year"]
specs += [("wh_est_year (recorded years)", "wh_est_year", recorded_year),
          ("wh_est_year (filled, as a model receives it)", "wh_est_year", pd.Series(True, index=df.index))]

rows = []
for label, c, use in specs:
    d = df[use]
    r = d["is_unrated_warehouse"] == 0
    groups = [g.to_numpy() for _, g in d.groupby("breakdown_risk", observed=True)[c]]
    medians = d.groupby("breakdown_risk", observed=True)[c].median()

    # eta squared — all warehouses
    _vals = d[c]; _groups = d["breakdown_risk"]
    _gm = _vals.mean()
    _between = sum(len(v) * (v.mean() - _gm) ** 2 for _, v in _vals.groupby(_groups, observed=True))
    effect = _between / ((_vals - _gm) ** 2).sum()

    # size label (thresholds 0.01 / 0.06 / 0.14)
    size_all = "large" if effect >= 0.14 else "medium" if effect >= 0.06 else "small" if effect >= 0.01 else "negligible"

    # eta squared — rated only
    _vals_r = d.loc[r, c]; _groups_r = d.loc[r, "breakdown_risk"]
    _gm_r = _vals_r.mean()
    _between_r = sum(len(v) * (v.mean() - _gm_r) ** 2 for _, v in _vals_r.groupby(_groups_r, observed=True))
    eta_rated = _between_r / ((_vals_r - _gm_r) ** 2).sum()

    rows.append({
        "feature": label, "warehouses_used": len(d),
        "anova_p": stats.f_oneway(*groups).pvalue, "kruskal_p": stats.kruskal(*groups).pvalue,
        "eta_sq_all": effect, "size_all": size_all,
        "eta_sq_rated_only": eta_rated,
        "spearman_with_class_all": d[c].corr(class_order[d.index], method="spearman"),
        "spearman_with_class_rated": d.loc[r, c].corr(class_order[d.index][r], method="spearman"),
        "median Not High Risk / High Risk": " / ".join(f"{m:g}" for m in medians.reindex(CLASSES)),
    })
numeric_tests = pd.DataFrame(rows).sort_values("eta_sq_all", ascending=False).reset_index(drop=True)
numeric_tests.assign(
    anova_p=numeric_tests["anova_p"].map("{:.3g}".format),
    kruskal_p=numeric_tests["kruskal_p"].map("{:.3g}".format),
).round(4)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for ax, (label, c, use) in zip(axes.ravel(), specs):
    sns.boxplot(data=df[use], x="breakdown_risk", y=c, order=CLASSES, hue="breakdown_risk", hue_order=CLASSES,
                palette=CLASS_COLOURS, legend=False, linewidth=1, fliersize=1.5, ax=ax)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel(""); ax.set_ylabel("")
for ax in axes.ravel()[len(specs):]:
    ax.axis("off")
fig.suptitle("Numeric features by breakdown-risk class", fontweight="bold", y=1.0)
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> - **Three numeric features differ between the binary risk classes; the other eight do not.**
>
> | Feature | η² all (size) | η² rated only | Median Not High Risk / High Risk |
> |---|---|---|---|
> | establishment year — recorded years | large | rated: medium | ~2011 / ~2007 |
> | storage issues | large | rated: medium | ~15 / ~20 |
> | shipment weight | large | rated: medium | ~20,000 / ~25,000 t |
> | establishment year — filled | medium | rated: small–medium | compressed by filling |
> | the other eight | negligible | negligible | identical across classes |
>
> - **High Risk warehouses are older, ship more and report more storage issues.** The boxplots show clear separation
>   on all three features, with High Risk shifted higher on shipment and storage issues, and lower on establishment year.
>
> - **Most of the signal survives without the unrated group.** Rated-only η² stays medium to large, so these
>   relationships belong to the operating network, not only to the 908 warehouses at zero breakdowns.
>
> - **Filling the missing years costs this objective signal** — the filled column a model receives has lower η² than the
>   recorded-year column. The boxplot shows why: filled warehouses collapse into a narrower band, with the true spread
>   pushed out as outlying points.
>
> - **Significant is not the same as important.** Transport issues may reach significance with η² near zero —
>   their median is 0 in both classes.
>
> - **Evidence for feature selection.** Of the five period measures, only **storage issues and shipment weight** carry any
>   signal; refills, government checks and transport issues carry none. Of the six numeric characteristics, only
>   **establishment year** does. The same-period signal and the characteristic signal are of similar strength, and §7
>   shows they overlap heavily.

---
## 6. Categorical and 0/1 features against risk class

This section applies chi-square tests. **Cramér's V** measures how strongly each feature and the binary breakdown risk
are associated, on a scale from 0 (none) to 1 (one determines the other). As in NB 15 §6, V is read against Cohen's
thresholds — 0.10, 0.30 and 0.50 — divided by √(m − 1), where m is the smaller dimension of the table, because V runs
lower for larger tables.

Again, all warehouses and rated warehouses only. `is_unrated_warehouse` cannot be tested on rated warehouses (it is 0 for
all of them), and among rated warehouses the certificate has five grades rather than six.

The last columns turn V into business terms: across the levels of a feature, the lowest and highest share of warehouses in
the **High Risk** class. The network's own High Risk share is printed beneath the heatmap for comparison.

In [ ]:
rows = []
for c in binary_features + categorical_features:
    # Cramér's V — all warehouses
    _table = pd.crosstab(df[c], df["breakdown_risk"])
    _chi2, p_all = stats.chi2_contingency(_table, correction=False)[:2]
    m_all = min(_table.shape)
    v_all = np.sqrt(_chi2 / (_table.to_numpy().sum() * (m_all - 1)))
    _sm, _md, _lg = (t / np.sqrt(m_all - 1) for t in (0.10, 0.30, 0.50))
    size_all = "large" if v_all >= _lg else "medium" if v_all >= _md else "small" if v_all >= _sm else "negligible"

    # Cramér's V — rated warehouses only
    if c == "is_unrated_warehouse":
        v_rated, m_rated, size_rated = np.nan, np.nan, "—"
    else:
        _table_r = pd.crosstab(df.loc[rated, c], df.loc[rated, "breakdown_risk"])
        _chi2_r, _ = stats.chi2_contingency(_table_r, correction=False)[:2]
        m_rated = min(_table_r.shape)
        v_rated = np.sqrt(_chi2_r / (_table_r.to_numpy().sum() * (m_rated - 1)))
        _sm_r, _md_r, _lg_r = (t / np.sqrt(m_rated - 1) for t in (0.10, 0.30, 0.50))
        size_rated = ("large" if v_rated >= _lg_r else "medium" if v_rated >= _md_r
                      else "small" if v_rated >= _sm_r else "negligible")

    high_share = 100 * pd.crosstab(df[c], df["breakdown_risk"], normalize="index")["High Risk"]
    rows.append({
        "feature": c, "kind": "0/1" if c in binary_features else "categorical",
        "chi2_p": p_all, "cramers_v_all": v_all,
        "size_all": size_all,
        "cramers_v_rated_only": v_rated,
        "size_rated_only": size_rated,
        "High Risk share, lowest level": f"{high_share.min():.1f}% ({high_share.idxmin()})",
        "High Risk share, highest level": f"{high_share.max():.1f}% ({high_share.idxmax()})",
    })
categorical_tests = pd.DataFrame(rows).sort_values("cramers_v_all", ascending=False).reset_index(drop=True)
categorical_tests.assign(chi2_p=categorical_tests["chi2_p"].map("{:.3g}".format)).round(4)

The heatmap shows, for every level, how far the share of each class **within that level** sits above or below the
network's share of that class, in percentage points. For 0/1 features only the level `= 1` is shown. Levels whose rows are
grey carry no information about risk. The colour scale is capped so that small differences stay visible; every cell carries
its exact figure.

In [ ]:
level_order = {"approved_wh_govt_certificate": ["C", "B", "B+", "A", "A+", "Unrated"],
               "WH_capacity_size": ["Small", "Mid", "Large"]}
network_share = 100 * df["breakdown_risk"].value_counts(normalize=True).reindex(CLASSES)

share_tables = []
for c in binary_features + categorical_features:
    t = 100 * pd.crosstab(df[c], df["breakdown_risk"], normalize="index")[CLASSES]
    if c in binary_features:
        t = t.loc[[1]]
    if c in level_order:
        t = t.reindex(level_order[c])
    t.index = [f"{c} = {level}" for level in t.index]
    share_tables.append(t)
class_shares = pd.concat(share_tables)
difference = class_shares.sub(network_share, axis=1).round(1) + 0.0

fig, ax = plt.subplots(figsize=(8, 12))
sns.heatmap(difference, annot=True, fmt="+.1f", cmap=DIVERGING, center=0, vmin=-15, vmax=15,
            linewidths=2, linecolor="white", annot_kws={"size": 8},
            cbar_kws={"label": "percentage points above (+) or below (−) the network's share of the class"}, ax=ax)
ax.set_title("Share of each risk class within each level, against the network\n(colour capped at ±15 pp; figures exact)",
             fontweight="bold")
ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.show()

print("network share of each class (%):", network_share.round(2).to_dict())

> **Interpretation.**
>
> - **The unrated flag is the strongest categorical signal, and it dominates the certificate's.**
>
> - `is_unrated_warehouse`: large V. No unrated warehouse is High Risk (all 908 have 0 breakdowns = Not High Risk).
>   In the heatmap its row and the certificate's `Unrated` row — the same 908 warehouses — are the saturated ones.
> - Certificate grade: large V across all warehouses, but **negligible among rated warehouses**.
>   Almost all of the certificate's association comes from its `Unrated` level.
>
> - **Among rated warehouses, grades show a small gradient.** A+ warehouses are somewhat more often High Risk;
>   C warehouses are somewhat less often High Risk. Because every unrated warehouse is Not High Risk, rated levels
>   generally show a higher High Risk share than the network average.
>
> - **Everything else is negligible for the binary target**, whether or not it reaches significance:
>   - **Location type:** Urban warehouses are slightly more often High Risk than rural (small V).
>   - **Temperature regulation, ownership and zone** are significant at 5% with very small V.
>   - **Regional zone, capacity size, both flood indicators and electric back-up** — not significant; shares move by
>     no more than a few percentage points.
>
> - **The recording flag carries no information about breakdown risk (binary).** `wh_est_year_missing` has negligible V
>   and its heatmap row is nearly flat. *Whether* the year was recorded is unrelated to risk — even though §5 showed
>   that filling it weakens the year's signal.

---
## 7. Multivariate — do features repeat each other, and are the classes separable?

**Repetition among features.** Logistic regression spreads credit unpredictably between features that move together,
which makes its coefficients unstable. Naive Bayes goes further and *assumes* features are independent within each class.
Two measures:

- **Spearman correlation** among the numeric and 0/1 features;
- the **variance inflation factor (VIF)** — how far each feature can be reproduced from all the others. A common rule of
  thumb reads above 5 as worth attention and above 10 as serious.

Categorical features are summarised by their strongest pairwise Cramér's V.

In [ ]:
corr_features = numeric_features + binary_features
spearman = df[corr_features].corr(method="spearman")

plt.figure(figsize=(12, 10))
sns.heatmap(spearman, annot=True, fmt=".2f", cmap=DIVERGING, center=0, vmin=-1, vmax=1,
            annot_kws={"size": 7}, linewidths=1, linecolor="white", cbar_kws={"label": "Spearman correlation"})
plt.title("Spearman correlation among numeric and 0/1 features\n(wh_est_year filled, as a model receives it)",
          fontweight="bold")
plt.tight_layout(); plt.show()

mask = np.triu(np.ones(spearman.shape, dtype=bool), k=1)
pairs = spearman.where(mask).stack().rename("spearman").reset_index()
pairs.columns = ["feature_a", "feature_b", "spearman"]
print("strongest pairs:")
print(pairs.reindex(pairs["spearman"].abs().sort_values(ascending=False).index).head(8).round(3).to_string(index=False))

design = sm.add_constant(df[corr_features].astype(float))
vif = pd.Series([variance_inflation_factor(design.to_numpy(), i) for i in range(1, design.shape[1])],
                index=corr_features, name="VIF").sort_values(ascending=False)
print("\nvariance inflation factor:")
print(vif.round(2).to_string())

rows = []
for i, a in enumerate(categorical_features):
    for b in categorical_features[i + 1:]:
        _table = pd.crosstab(df[a], df[b])
        _chi2, _ = stats.chi2_contingency(_table, correction=False)[:2]
        _m = min(_table.shape)
        _v = np.sqrt(_chi2 / (_table.to_numpy().sum() * (_m - 1)))
        rows.append({"feature_a": a, "feature_b": b, "cramers_v": round(_v, 3)})
print("\nstrongest associations between categorical features:")
print(pd.DataFrame(rows).sort_values("cramers_v", ascending=False).head(4).to_string(index=False))

> **Interpretation.**
>
> - **One pair repeats itself; nothing else does.** Shipment weight and storage issues correlate at **0.989**, and their VIFs
>   are **67.47** and **62.38** — far past the "serious" line of 10. Every other feature's VIF is **1.77 or less**. Read from
>   the heatmap: apart from that pair, the matrix is mostly near-zero grey. The strongest remaining colour is the year's
>   link with storage issues (−0.615) and shipment (−0.594), refills with the recording flag (−0.551), and workers with
>   electric back-up (0.393).
>
> - For **logistic regression**, keeping both members of the shipment–storage-issues pair would make their coefficients
>   unstable. For **Naive Bayes**, it would count the same evidence twice. Tree models are affected far less.
> - **Refills against the recording flag (−0.551)** is the pattern NB 01 §4 found: the year is missing for every warehouse
>   with 0–2 refill requests.
> - Among the categorical features, **capacity size and regional zone are near-duplicates (V 0.847)**. Both are
>   negligible against the binary target (§6), so the pair matters only for how much noise they add. Every other
>   categorical pair is 0.178 or less.

**Are the classes separable?** A classifier needs the classes to occupy at least partly different regions of the feature
space. Principal component analysis reduces the numeric and 0/1 features (standardised, so that no unit dominates) to the
two directions of greatest variation. The two binary classes are then drawn as **separate panels on identical axes**, not
overlaid, so that neither class hides the other. A fixed sample of 6,000 warehouses keeps the points readable. This is a
view, not a test: classes can be separable in directions this map does not show.

In [ ]:
X_std = StandardScaler().fit_transform(df[corr_features])
pca = PCA(n_components=2, random_state=RANDOM_STATE).fit(X_std)
scores = pd.DataFrame(pca.transform(X_std), columns=["PC1", "PC2"], index=df.index)
scores["breakdown_risk"] = df["breakdown_risk"]

print(f"variance explained: PC1 {100 * pca.explained_variance_ratio_[0]:.2f}%   "
      f"PC2 {100 * pca.explained_variance_ratio_[1]:.2f}%\n")
loadings = pd.DataFrame(pca.components_.T, index=corr_features, columns=["PC1", "PC2"])
print("features loading most strongly on each component:")
for pc in ["PC1", "PC2"]:
    top = loadings[pc].abs().sort_values(ascending=False).head(4).index
    print(f"  {pc}: " + ", ".join(f"{f} ({loadings.loc[f, pc]:+.2f})" for f in top))

sample = scores.sample(6_000, random_state=RANDOM_STATE)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True, sharey=True)
for ax, k in zip(axes, CLASSES):
    ax.scatter(sample["PC1"], sample["PC2"], s=4, color="#e1e0d9", linewidth=0)
    part = sample[sample["breakdown_risk"] == k]
    ax.scatter(part["PC1"], part["PC2"], s=5, alpha=0.6, color=CLASS_COLOURS[k], linewidth=0)
    centre = scores.loc[scores["breakdown_risk"] == k, ["PC1", "PC2"]].mean()
    ax.scatter(*centre, s=120, marker="X", color="#0b0b0b")
    ax.set_title(f"{k} ({len(part):,} of the 6,000 sampled)", fontweight="bold")
    ax.set_xlabel("PC1")
axes[0].set_ylabel("PC2")
fig.suptitle("Warehouses on the first two principal components — each binary class against all others (grey); "
             "× = class centre (all warehouses)", fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()

print("class centres on PC1 / PC2 (all warehouses):")
print(scores.groupby("breakdown_risk", observed=True)[["PC1", "PC2"]].mean().round(3).to_string())

far_left = scores["PC1"] < -3
print(f"\nwarehouses at PC1 below -3 (all 25,000): {int(far_left.sum()):,}; of these unrated: "
      f"{int(df.loc[far_left, 'is_unrated_warehouse'].sum()):,}; unrated warehouses elsewhere: "
      f"{int(df.loc[~far_left, 'is_unrated_warehouse'].sum()):,}")
print("risk class of the warehouses at PC1 below -3:",
      df.loc[far_left, "breakdown_risk"].value_counts().reindex(CLASSES).to_dict())

> **Interpretation.**
>
> - The first two components hold only **15.97%** and **10.23%** of the variance. Most features vary
>   independently, so no two directions summarise the data — this map is a partial view.
>
> - **PC1 is the volume–age direction** (storage issues +0.58, shipment weight +0.57, establishment year −0.45, unrated
>   −0.28). **PC2 is the recording and refill direction** (recording flag +0.63, refills −0.61, temperature regulation
>   −0.36).
> - **Read from the panels: the two classes share much of the same cloud, but High Risk is shifted to the right.**
>   Not High Risk occupies the left part of the cloud (lower PC1), including a stripe at the far left that is exactly
>   the 908 unrated warehouses — all with 0 breakdowns and all Not High Risk. High Risk is shifted toward higher PC1
>   (higher volume and older age). The two panels overlap substantially, reflecting that the 3|4 step, while the
>   strongest remaining separator, is weaker than the early 0|1 and 1|2 steps.
>
> - The 908 unrated warehouses at far-left PC1 are all Not High Risk, and this sub-cluster would be trivially
>   identified by any model. Beyond them, the two classes overlap across most of the map.

---
## 8. What the evidence means for the classification design

The binary target is confirmed on the training split in the transformation step. From §3.3 onward, every section uses the
adopted binary target (Not High Risk: 0–3 breakdowns, High Risk: 4–6 breakdowns).

**Feature scope.** The main models use all 23 candidate features. The best model is also refitted on characteristics only —
excluding the five period measures counted over the same window as the breakdowns — and the two results are compared in the
results notebook. This lets the project answer whether a preventive-planning model can rest on observable warehouse
properties alone.

**Open items carried to later steps.**

- *Duplicate pair:* shipment weight and storage issues correlate at ρ 0.989 (VIF 67.47 / 62.38); the feature engineering
  step decides which to keep, on training data, using probe models.
- *Establishment year:* filling the missing years reduces the year's signal (η² 0.2333 recorded vs 0.1223 filled); the
  recording flag carries almost no information by itself (V 0.0045); the transformation step decides whether to keep the
  flag alongside the filled year.
- *Encoding and scaling:* standard-deviation ratio 10,164 : 1; 23 one-hot columns. The transformation step specifies
  encoding and scaling rules.
- *Negligible features:* eight numeric features and ten categorical and binary features carry no one-at-a-time signal;
  capacity size and regional zone are near-duplicates (V 0.847). The feature engineering step tests whether removing them
  as a group hurts probe-model performance.
- *Class balance:* 1.09 : 1 — the split is near-balanced, so no resampling is expected to be necessary, but this is
  verified on the training split.

**Open questions for modelling.**

- Which model best separates High Risk from Not High Risk?
- Does the near-balanced 1.09 : 1 split mean no resampling is needed?
- How well do the two classes separate on the key features (shipment weight, storage issues, establishment year)?

---
## 9. Checks

This notebook is exploratory. Apart from the target label it built in memory, it must not have altered the data it read,
and it writes nothing.

In [ ]:
assert df.drop(columns="breakdown_risk").equals(load_preprocessed()), "the data was modified — this notebook must only report"
assert df.shape == (25_000, 26)
assert list(df["breakdown_risk"].cat.categories) == CLASSES and df["breakdown_risk"].cat.ordered
print(f"data unchanged: {len(df):,} rows; only the in-memory target label was added; nothing written by this notebook")

---
## Summary

**Binary target.**

- The breakdown count is split into two classes at the 3|4 boundary.
- **Not High Risk**: 0–3 breakdowns — 13,026 warehouses (52.1%).
- **High Risk**: 4–6 breakdowns — 11,974 warehouses (47.9%).
- The split is near-balanced at 1.09 : 1. Accuracy is a safe primary metric.
- The 3|4 step carries the most signal across all three key features (η² 0.024–0.035).
- The naive alternative — any breakdown vs none — captures only the unrated-warehouse rule and is analytically useless.

**What predicts breakdown risk (binary) — as association, since the data is a single snapshot.**

- **Three numeric features, all strongly:** establishment year (large η² on recorded years), storage issues (large η²)
  and shipment weight (large η²).
- High Risk warehouses are older, ship more and log more storage issues.
- **One on/off feature:** the unrated flag (large V). All 908 unrated warehouses are Not High Risk.
- Certificate grades add almost nothing among rated warehouses (negligible V).
- **Nothing else is useful enough for a business claim:** refills, government checks, transport issues, staffing,
  distance, competitors, retail shops, distributors, location, zone, capacity, ownership, flood exposure, electric
  back-up, temperature regulation, and whether the year was recorded.

**What the model-building stages must handle.**

- Shipment weight and storage issues are near-duplicates (VIF 67.47 / 62.38) — carried to the feature engineering step.
- Filling the missing years reduces their signal — carried to the transformation step.
- Scales differ 10,164-fold — carried to the transformation step.
- Most features carry no one-at-a-time signal — carried to the feature engineering step.
- Classes are near-balanced (1.09 : 1) — resampling is not expected, but confirmed on the training split.

**Feature scope.**

- Main models use all candidate features.
- The best model is refitted on characteristics only.
- The two results are compared because five measures are counted over the same period as the breakdowns.

**Handed to the transformation step.**

- Binary target banding (Not High Risk: 0–3, High Risk: 4–6).
- Feature roles.
- Open items: establishment year representation, encoding, scaling, duplicate pair.